# Лаборатория 1.3. Ответ по форме и вызов инструментов

**Что мы сделаем:** заставим модель отвечать строго по заданной форме (чтобы её ответ
могла прочитать программа, а не человек), а потом дадим ей настоящий инструмент и
посмотрим, как она его просит вызвать. Это и есть механизм, на котором построены
все ИИ-агенты.

**Что понадобится:** код класса от учителя — как в прошлой лаборатории.

In [ ]:
!pip -q install openai

In [ ]:
import getpass
import json
import os
from pprint import pprint

from openai import OpenAI

ADRES = "https://ai9.adelfos.ru/api/v1"
# Здесь модель указана жёстко: не все модели одинаково хорошо заполняют параметры
# инструментов, а эта справляется. В конце лаборатории проверим и другие.
MODEL = "qwen/qwen3.7-flash"

try:
    from google.colab import userdata
    KOD_KLASSA = userdata.get("AI9_KOD") or os.environ.get("AI9_KOD")
except Exception:
    KOD_KLASSA = os.environ.get("AI9_KOD")

# Сервер проверит код, только когда мы обратимся к нему с ключом. Поэтому делаем один
# лёгкий запрос (список моделей) и, если код не принят, спрашиваем его заново.
client = None
while client is None:
    if not KOD_KLASSA:
        KOD_KLASSA = getpass.getpass("Код класса: ")
    client = OpenAI(base_url=ADRES, api_key=KOD_KLASSA)
    try:
        client.models.list()   # неверный код сервер не примет и ответит ошибкой
        print("Всё хорошо: код подошёл. Модель:", MODEL)
    except Exception:
        print("Код не подошёл — проверь его у учителя и введи заново.")
        client = None
        KOD_KLASSA = None      # после ошибки код из секретов и окружения больше не берём

## Шаг 1. Проблема: программа не умеет читать по-человечески

Представь школьного бота, который разбирает объявления и складывает их в расписание.
Ему нужно достать из текста четыре вещи: название, дни, время и кабинет.

Попросим модель словами: «разбери в JSON». JSON — это способ записать данные так,
чтобы их понимала программа: пары «имя поля — значение» в фигурных скобках.

In [ ]:
OBJAVLENIE = "Кружок робототехники, вторник и четверг в 15:40, кабинет 204"

zapros = [{"role": "user", "content":
    f"Разбери объявление в JSON с полями nazvanie, dni, vremya, kabinet.\n\n{OBJAVLENIE}"}]

print("Что мы отправляем модели:")
pprint(zapros, width=100, sort_dicts=False)

otvet = client.chat.completions.create(
    model=MODEL,
    messages=zapros,
    temperature=0,
    max_tokens=250,
)

tekst = otvet.choices[0].message.content

print("\nЧто нам ответило:")
print(repr(tekst))     # repr показывает строку «как есть», со всеми служебными символами

Посмотри на вывод внимательно — там два сюрприза.

**Сюрприз первый.** Ответ обёрнут в <code>```json … ```</code> — это оформление для
человека, для красивой подсветки. Программе такое не подходит: попытка прочитать это
как JSON закончится ошибкой. Убедимся:

In [ ]:
try:
    dannye = json.loads(tekst)
    print("Прочиталось:", dannye)
except json.JSONDecodeError as oshibka:
    print("Не прочиталось! Ошибка:", oshibka)
    print("Придётся руками отрезать обёртку — и надеяться, что в следующий раз")
    print("модель обернёт ответ точно так же, а не иначе.")

**Сюрприз второй.** Даже если обёртку убрать, кабинет пришёл строкой `"204"`, а не
числом `204`. Для программы это разные вещи: со строкой нельзя сравнить «больше/меньше»,
нельзя посчитать, а попытка сложить «204» + 1 даст ошибку или «2041».

Вывод: **просьба словами не даёт гарантии.** Сегодня модель ответила так, завтра иначе.

## Шаг 2. Решение: выдать модели бланк

Вместо просьбы можно передать **схему** — точное описание, какие поля должны быть
и какого они типа. Модель тогда физически не может ответить иначе: сервер модели
следит за формой во время генерации.

Разберём схему по частям:

* `"type": "object"` — ответ должен быть объектом (тем самым «в фигурных скобках»);
* `properties` — перечень полей и их типов (`string` — текст, `integer` — целое число,
  `array` — список);
* `required` — какие поля обязательны;
* `additionalProperties: False` — лишних полей быть не должно;
* `strict: True` — соблюдать схему строго.

In [ ]:
SHEMA = {
    "type": "object",
    "properties": {
        "nazvanie": {"type": "string"},
        "dni": {"type": "array", "items": {"type": "string"}},
        "vremya": {"type": "string"},
        "kabinet": {"type": "integer"},
    },
    "required": ["nazvanie", "dni", "vremya", "kabinet"],
    "additionalProperties": False,
}

zapros = [{"role": "user", "content": f"Разбери объявление.\n\n{OBJAVLENIE}"}]

print("Что мы отправляем модели:")
pprint(zapros, width=100, sort_dicts=False)

otvet = client.chat.completions.create(
    model=MODEL,
    messages=zapros,
    response_format={"type": "json_schema",
                     "json_schema": {"name": "kruzhok", "strict": True, "schema": SHEMA}},
    temperature=0,
    max_tokens=250,
)

tekst = otvet.choices[0].message.content
print("\nСырой ответ:", repr(tekst))

dannye = json.loads(tekst)          # теперь читается без всяких ухищрений
print()
print("Название:", dannye["nazvanie"])
print("Дни:     ", dannye["dni"])
print("Кабинет: ", dannye["kabinet"], "— тип:", type(dannye["kabinet"]).__name__)

Сравни с шагом 1: обёртки нет, кабинет — **число**, поля ровно те, что заказывали.
Теперь с этим можно работать: сложить расписание, отсортировать, положить в базу.

> **Структурированный ответ** — ответ модели по заранее заданной форме (схеме),
> который программа читает напрямую, без разбора человеческого языка.

Это первая половина сегодняшней лаборатории. Вторая — то, ради чего она затевалась.

## Шаг 3. Инструменты: как модель «делает» то, чего не умеет

Модель умеет только писать текст. Она не может посмотреть погоду, посчитать большое
выражение или заглянуть в школьную базу. Зато она может **попросить** программу это
сделать — если заранее объяснить ей, какие просьбы бывают.

Опишем инструмент. Описание — та же схема, что и выше, плюс имя и объяснение,
**когда** им пользоваться. Это объяснение модель читает как обычный текст, поэтому
писать его нужно понятно: от него зависит, догадается ли она вызвать инструмент.

In [ ]:
INSTRUMENTY_OPISANIE = [{
    "type": "function",
    "function": {
        "name": "uznat_pogodu",
        "description": "Узнать текущую погоду в указанном городе",
        "parameters": {
            "type": "object",
            "properties": {
                "gorod": {"type": "string", "description": "Название города, например «Москва»"},
            },
            "required": ["gorod"],
            "additionalProperties": False,
        },
    },
}]

# А это сам инструмент — обычная функция. Никакого ИИ внутри.
# Настоящая ходила бы в интернет; нам для урока хватит игрушечной таблички.
POGODA = {"Москва": "+5, облачно", "Сочи": "+18, солнечно", "Норильск": "-24, метель"}


def uznat_pogodu(gorod):
    return POGODA.get(gorod, f"нет данных по городу {gorod}")

Теперь зададим вопрос, на который модель сама ответить не может, и передадим ей
описание инструмента. Важно: мы **не просим** её вызвать инструмент — она решает сама.

In [ ]:
istoriya = [{"role": "user", "content": "Какая сейчас погода в Москве?"}]

print("Что мы отправляем модели (история разговора):")
pprint(istoriya, width=100, sort_dicts=False)

otvet = client.chat.completions.create(
    model=MODEL,
    messages=istoriya,
    tools=INSTRUMENTY_OPISANIE,
    temperature=0,
    max_tokens=200,
)

print()
soobshchenie = otvet.choices[0].message
print("Текст ответа:   ", soobshchenie.content)
print("Причина остановки:", otvet.choices[0].finish_reason)
print()
for vyzov in soobshchenie.tool_calls or []:
    print("Модель просит вызвать:", vyzov.function.name)
    print("С параметрами:        ", vyzov.function.arguments)

**Останови взгляд здесь.** Обрати внимание на три вещи:

1. `content` — пустой (`None`). Текста для человека нет вообще.
2. `finish_reason` — не `stop`, а `tool_calls`: модель остановилась не потому, что
   договорила, а потому что ждёт результата.
3. В `tool_calls` лежит имя функции и параметры — **в виде текста**. Модель по-прежнему
   только пишет текст! Просто этот текст оформлен как заявка на вызов.

> **Инструмент** — обычная функция в программе, которую модели разрешили попросить
> вызвать. Выполняет её программа, а не модель.

## Шаг 4. Замыкаем круг

Модель попросила — значит, надо выполнить и вернуть результат. Это делает наш код:
читает заявку, вызывает настоящую функцию, кладёт ответ обратно в историю разговора
и снова обращается к модели. Теперь у неё есть факт, и она может ответить человеку.

In [ ]:
if soobshchenie.tool_calls:
    vyzov = soobshchenie.tool_calls[0]

    # 1. Разбираем заявку. Параметры приходят текстом, поэтому читаем их как JSON.
    argumenty = json.loads(vyzov.function.arguments or "{}")
    gorod = argumenty.get("gorod")
    print("1) Модель попросила погоду для города:", gorod)

    # Модель могла забыть параметр — программа обязана это пережить, а не упасть.
    if not gorod:
        print("   Параметра нет — уточняем у модели или подставляем значение по умолчанию.")
        gorod = "Москва"

    # 2. Выполняем НАСТОЯЩУЮ функцию.
    rezultat = uznat_pogodu(gorod)
    print("2) Программа выполнила инструмент, результат:", rezultat)

    # 3. Кладём в историю и заявку модели, и результат инструмента.
    istoriya.append(soobshchenie)
    istoriya.append({
        "role": "tool",                       # особая роль: «это ответ инструмента»
        "tool_call_id": vyzov.id,             # к какой именно заявке относится
        "content": rezultat,
    })

    # 4. Спрашиваем модель снова — теперь ей есть на что опереться.
    print("3) Полная история, которую мы отправляем модели на последнем шаге:")
    pprint(istoriya, width=100, sort_dicts=False)
    print()
    final = client.chat.completions.create(
        model=MODEL, messages=istoriya, tools=INSTRUMENTY_OPISANIE,
        temperature=0, max_tokens=200,
    )
    print("3) Финальный ответ человеку:", final.choices[0].message.content)

Посмотри на порядок сообщений в `istoriya` — это и есть весь «искусственный интеллект»:

```
user      : Какая сейчас погода в Москве?
assistant : (текста нет) хочу вызвать uznat_pogodu(gorod="Москва")
tool      : +5, облачно
assistant : В Москве сейчас +5 и облачно.
```

Ни на одном шаге модель никуда не ходила и ничего не делала. Она дважды написала текст,
а всё остальное сделала обычная программа.

> **Агент** — программа, которая крутит этот круг не один раз, а столько, сколько нужно:
> модель просит инструмент, код выполняет, модель смотрит на результат и решает, что дальше.

Целиком этим займёмся в теме 4 — там же поставим агенту границы, чтобы он не крутился вечно.

## Попробуй сам

1. Спроси погоду в городе, которого нет в `POGODA` (например, в Казани). Что ответит
   модель, получив «нет данных»? Признается или выдумает?
2. Убери у инструмента `description` (оставь пустую строку) и задай вопрос снова.
   Догадается ли модель его вызвать? Так ты поймёшь, зачем описание нужно.
3. Спроси что-нибудь, для чего инструмент не нужен («сколько будет 2+2»). Модель
   вызовет инструмент или ответит сама?
4. Поменяй `MODEL` на `google/gemma-3-12b-it` и повтори шаг 3. Скорее всего, она
   попросит инструмент, но **забудет заполнить параметр** — как раз тот случай,
   ради которого в шаге 4 стоит проверка «если параметра нет».

## Что унести с собой

* Просьба «ответь в JSON» словами ненадёжна: приходит обёртка <code>```json</code>,
  а числа иногда становятся строками.
* **Схема** (`response_format`) даёт гарантию: те поля, те типы, без лишнего.
* **Инструмент** — обычная функция; модель только просит её вызвать, выполняет программа.
* При вызове инструмента `content` пустой, а `finish_reason` равен `tool_calls`.
* Результат инструмента возвращают в историю с ролью `tool` — и спрашивают модель снова.
* Модель может ошибиться в параметрах, поэтому код обязан проверять то, что она прислала.